# ch10. 직교성과 투영 — '가장 가까운 답'을 찾는 법 📏

웹 챕터 10에서 배운 내용을 NumPy 로 직접 계산해 봅니다.

**이 노트북에서 하는 것**
1. **직교** 판별: 내적이 0인지 확인
2. **투영(그림자)** 공식을 직접 구현하고, **잔차가 수직**임을 숫자로 확인
3. **최소제곱**: 해가 없는 문제에 `np.linalg.lstsq` 로 추세선(선형회귀) 맞추기

> 용어 복습. **직교**: 수직, 내적 = 0. **투영**: 벡터가 다른 방향에 드리운 그림자
> $\vec{p} = \frac{\vec{a}\cdot\vec{b}}{\vec{b}\cdot\vec{b}}\vec{b}$. **잔차**: 그림자로 설명 못 한 나머지 $\vec{a}-\vec{p}$ (항상 $\vec{b}$ 와 수직).
> **최소제곱**: 잔차 제곱합이 최소가 되는 답 = 투영으로 찾는 답.

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch10')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (그래프의 한글이 깨지지 않게)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('numpy', np.__version__, '준비 완료')

## 1. 직교 판별: 내적이 0인가

판별법은 이것 하나: $\vec{u} \perp \vec{v} \iff \vec{u}\cdot\vec{v} = 0$

In [ ]:
pairs = {
    'u=(2,1),  v=(-1,2)': (np.array([2, 1]), np.array([-1, 2])),
    'u=(3,0),  v=(0,5) ': (np.array([3, 0]), np.array([0, 5])),
    'u=(2,1),  v=(1,1) ': (np.array([2, 1]), np.array([1, 1])),
}

for name, (u, v) in pairs.items():
    d = np.dot(u, v)
    print(f'{name}: 내적 = {d:2d} ->', '직교(수직)! ⊥' if d == 0 else '직교 아님')

## 2. 투영: 그림자 계산하고 그림으로 확인

$$\vec{p} = \operatorname{proj}_{\vec{b}}(\vec{a}) = \underbrace{\frac{\vec{a}\cdot\vec{b}}{\vec{b}\cdot\vec{b}}}_{c\ (몇\ 배\ 지점)}\ \vec{b}$$

내적 두 번이면 끝. 그리고 **잔차 $\vec{a}-\vec{p}$ 와 $\vec{b}$ 의 내적이 0**(수직)인지 꼭 확인해 봅니다.

In [ ]:
def project(a, b):
    """a 를 b 방향에 투영한 그림자 벡터를 돌려준다."""
    c = np.dot(a, b) / np.dot(b, b)   # 그림자가 b 의 몇 배 지점인가
    return c * b

a = np.array([2.0, 4.0])
b = np.array([4.0, 1.0])

p = project(a, b)      # 그림자
r = a - p              # 잔차 (설명 못 한 나머지)

print('그림자 p       =', p.round(3))
print('잔차 r = a - p =', r.round(3))
print('잔차 · b       =', round(np.dot(r, b), 10), ' <- 0 = 수직! 투영이 최선이라는 증명서')

In [ ]:
plt.figure(figsize=(5, 5))
# a, b, p 화살표
plt.quiver([0, 0, 0], [0, 0, 0], [a[0], b[0], p[0]], [a[1], b[1], p[1]],
           angles='xy', scale_units='xy', scale=1,
           color=['#6366f1', '#f59e0b', '#10b981'])
# 잔차: 그림자 끝에서 a 끝까지 (점선)
plt.plot([p[0], a[0]], [p[1], a[1]], '--', color='#94a3b8', label='잔차 (b 와 수직)')
# b 방향 직선 (그림자가 놓이는 바닥)
t = np.linspace(-1, 2, 10)
plt.plot(t * b[0], t * b[1], ':', color='#fbbf24', lw=1, label='b 방향 직선')

plt.text(a[0], a[1], ' a', color='#6366f1', fontsize=12, fontweight='bold')
plt.text(b[0], b[1], ' b', color='#f59e0b', fontsize=12, fontweight='bold')
plt.text(p[0], p[1] - 0.4, ' p (그림자)', color='#10b981', fontsize=11, fontweight='bold')
plt.axhline(0, color='gray', lw=0.5)
plt.axvline(0, color='gray', lw=0.5)
plt.xlim(-2, 6); plt.ylim(-2, 6)
plt.gca().set_aspect('equal')
plt.title('투영: a 가 b 방향에 드리운 그림자 p')
plt.legend(loc='upper left')
plt.show()

## 3. 최소제곱: 점 구름에 추세선 맞추기 (선형회귀)

공부 시간 $x$ 와 시험 점수 $y$ 데이터 30개에 직선 $y = ax + b$ 를 맞춰봅니다.

점 30개를 전부 지나는 직선은 없으므로($A\vec{x}=\vec{b}$ 의 해 없음),
**오차 제곱합이 최소**가 되는 $\hat{a}, \hat{b}$ 를 찾습니다. 원리는 **투영**이고,
NumPy 에서는 `np.linalg.lstsq` 한 줄이에요.

In [ ]:
rng = np.random.default_rng(42)

# 가짜 데이터: 점수 = 8 x 공부시간 + 40 + 잡음
hours = rng.uniform(0, 6, size=30)
score = 8 * hours + 40 + rng.normal(0, 6, size=30)

# A @ [기울기, 절편] = score 꼴로 세우기 (열: 공부시간, 상수 1)
A = np.column_stack([hours, np.ones_like(hours)])

coef, residual_ss, rank, sv = np.linalg.lstsq(A, score, rcond=None)
slope, intercept = coef
print(f'최소제곱 답: 점수 ≈ {slope:.2f} x 공부시간 + {intercept:.2f}')

# 같은 답을 정규방정식 (A^T A) x = A^T b 로도 확인
coef2 = np.linalg.solve(A.T @ A, A.T @ score)
print('정규방정식 답:', coef2.round(2), ' <- lstsq 와 동일!')

In [ ]:
xs = np.linspace(0, 6, 50)

plt.figure(figsize=(6, 4))
plt.scatter(hours, score, s=25, color='#6366f1', alpha=0.7, label='데이터 (학생 30명)')
plt.plot(xs, slope * xs + intercept, color='#ef4444', lw=2,
         label=f'최소제곱 직선 y = {slope:.1f}x + {intercept:.1f}')
# 잔차(각 점에서 직선까지 세로 거리)를 몇 개만 표시
for i in range(0, 30, 6):
    plt.plot([hours[i], hours[i]], [score[i], slope * hours[i] + intercept],
             ':', color='#94a3b8', lw=1)
plt.xlabel('공부 시간')
plt.ylabel('시험 점수')
plt.title('선형회귀 = 최소제곱 = 투영')
plt.legend()
plt.show()

print('점선(잔차)들의 제곱합이 최소가 되는 직선입니다.')

직선이 못 지나는 점들 사이에서 **잔차 제곱합이 최소**인 위치에 자리 잡았어요.
이게 벡터로는 "score 벡터를 A 열들의 span(ch9) 에 **투영**한 것"과 정확히 같은 계산입니다.

---

### 정리

| 개념 | 한 줄 요약 | NumPy |
|------|-----------|-------|
| 직교 | 수직 = 내적 0 | `np.dot(u, v) == 0` |
| 투영 | b 방향 그림자 | `(a@b)/(b@b) * b` |
| 잔차 | 설명 못 한 나머지, 항상 b 와 수직 | `a - p` |
| 최소제곱 | 잔차 제곱합 최소 = 투영 | `np.linalg.lstsq(A, b)` |
| 정규방정식 | 최소제곱의 공식 버전 | `solve(A.T@A, A.T@b)` |

**AI 연결**: 선형회귀(지도학습의 원형) · PCA 의 직교 축(중복 없는 요약) ·
임베딩에서 특정 방향 성분 제거(투영 빼기) · 그람-슈미트/QR(수치 안정성).

**다음 챕터(ch11)**: 특이값분해 SVD. ML에서 가장 중요한 분해입니다.